# MNIST 12×12 — Bond-Dimension Study

Figures built **straight from the analysis CSVs** of `mnist_full_r12` (MNIST resized to
12×12, Legendre embedding, `in_dim=3`, complex64, 64-class). No checkpoints are loaded —
the trained models live on mathqi and only the post-hoc results were transferred back.

- **Part 1** compares the analysed bond dimensions (**r10 / r20 / r40**).
- **Part 2** commits to one bond dimension (`BOND_DIM`, default **20**) and produces the
  alpha curves, defense bar plots, Gibbs comparisons, purification-radius comparison,
  the accept-rate curve and the model-comparison tables.

| | |
|---|---|
| Data | `analysis/outputs/mnist_full_r12/{nat,at}/legendre/d3r{10,20,40}c64/seed_sweep/{α}_{DDMM}/` |
| Figures | `figures/mnist_r12/compare/` (Part 1) · `figures/mnist_r12/{arch}/` (Part 2) |
| Seeds | 5 per (bond dim, model) |
| Attack | PGD-$\infty$, absolute $\varepsilon \in \{0.1, 0.2, 0.3\}$ in the model domain |
| Defenses | fixed absolute radius $\delta=0.2$ for AT, likelihood purification, and the local Gibbs move; Gibbs $k \in \{1,3,6\}$ and likelihood detection $q \in \{1,5,10,20\}\%$ |

**Prerequisites**: `conda activate bm4tc`, launch Jupyter from the repo root, run top to bottom.

To re-do Part 2 at another bond dimension, change `BOND_DIM` in §0.2 and re-run — every
figure path re-keys automatically.

---
# §0  Setup

In [ ]:
import sys
from pathlib import Path

# This notebook lives in notebooks/; Jupyter's CWD is that folder. Walk up to the
# repo root (the dir containing src/) so all analysis/figures paths resolve.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

In [ ]:
import matplotlib

# Paper styling: Times New Roman via LaTeX (mathptmx). Requires a system LaTeX install
# (latex + dvipng); set USE_LATEX = False to fall back to matplotlib mathtext.
USE_LATEX = True
if USE_LATEX:
    matplotlib.rcParams.update({
        "text.usetex": True,
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times"],
        "text.latex.preamble": r"\usepackage{mathptmx}",
    })
else:
    matplotlib.rcParams.update({"text.usetex": False, "font.family": "sans-serif"})

# Shared type sizes for every figure below (replaces per-cell LABEL_FS/TICK_FS/LEG_FS).
matplotlib.rcParams.update({
    "axes.labelsize": 15, "axes.titlesize": 15,
    "xtick.labelsize": 12, "ytick.labelsize": 12,
    "legend.fontsize": 11, "figure.dpi": 110, "savefig.dpi": 150,
})

# '%' must be escaped in BOTH engines: it is a comment char under usetex, and mathtext's
# parser rejects a bare '%' inside $...$. Always used inside math mode below.
PCT = r"\%"
print(f"USE_LATEX = {USE_LATEX}")

---
## §0.1  Registry — discover every analysed run

Scans the **consolidated** layout

```
analysis/outputs/mnist_full_r12/{nat,at}/legendre/d3r{R}c64/seed_sweep/{α-token}_{DDMM}/
    evaluation_data.csv    one row per seed — acc, rob/{ε}, uq_* (detection + likelihood purification)
    gibbs_data.csv         one row per seed — gibbs_* (Gibbs purification, k ∈ {1,3,6})
```

and builds two tidy long frames, `EVAL` and `GIBBS`, each keyed by
`(regime, arch, bond_dim, alpha)`.

Three things to know about this data:

1. **α is parsed from the directory name.** The `config/trainer.generative.criterion.kwargs.alpha`
   column is `NaN` in every consolidated CSV, so the `a0 / a001 / a01 / a02 / a05 / a1` token is
   the only source of truth.
2. **The older flat dirs are excluded** (`d3r20c64/seed_sweep_a0_1206`, …). They are superseded:
   one purification radius only and essentially no Gibbs data.
3. **Gibbs is evaluated on a 250-sample subsample** (`n_eval=250`, sampling error ≈ ±3%), while
   every `EVAL` metric uses the full test split. The two are *not* comparable at face value —
   this is repeated wherever they share an axis.

The coverage table printed below reports **effective** seed counts (rows with a non-NaN value),
so partially-failed runs are visible rather than silently averaged away.

In [ ]:
import re
import numpy as np
import pandas as pd

DATASET, EMBEDDING = "mnist_full_r12", "legendre"
ANALYSIS_ROOT = PROJECT_ROOT / "analysis" / "outputs" / DATASET
FIG_ROOT = PROJECT_ROOT / "figures" / "mnist_r12"

# Directory token -> alpha. The CSV's alpha config column is NaN, so this is authoritative.
ALPHA_TOKEN = {"a0": 0.0, "a001": 0.01, "a01": 0.1, "a02": 0.2, "a05": 0.5, "a1": 1.0}
ARCH_RE = re.compile(r"^d(\d+)r(\d+)c(\d+)$")


def discover():
    """[{regime, arch, bond_dim, alpha, dir}] for every consolidated seed-sweep run dir."""
    found = []
    for regime in ("nat", "at"):
        base = ANALYSIS_ROOT / regime / EMBEDDING
        if not base.is_dir():
            continue
        for arch_dir in sorted(base.iterdir()):
            m = ARCH_RE.match(arch_dir.name)
            if not m:
                continue
            for run_dir in sorted((arch_dir / "seed_sweep").glob("*_*")):
                token = run_dir.name.rsplit("_", 1)[0]
                if token not in ALPHA_TOKEN or not run_dir.is_dir():
                    continue
                found.append({"regime": regime, "arch": arch_dir.name,
                              "bond_dim": int(m.group(2)), "alpha": ALPHA_TOKEN[token],
                              "dir": run_dir})
    return found


RUNS = discover()


def _stack(filename):
    """Concatenate one CSV kind across all runs, prefixed with the registry keys."""
    frames = []
    for r in RUNS:
        p = r["dir"] / filename
        if not p.exists():
            continue
        df = pd.read_csv(p)
        df = df.assign(regime=r["regime"], arch=r["arch"], bond_dim=r["bond_dim"],
                       alpha=r["alpha"], run_dir=str(r["dir"]))
        keys = ["regime", "arch", "bond_dim", "alpha", "run_dir"]
        frames.append(df[keys + [c for c in df.columns if c not in keys]])
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


EVAL = _stack("evaluation_data.csv")
GIBBS = _stack("gibbs_data.csv")

print(f"{len(RUNS)} run dirs   EVAL {EVAL.shape}   GIBBS {GIBBS.shape}")
print(f"bond dims : {sorted(EVAL['bond_dim'].unique())}")
print(f"regimes   : {sorted(EVAL['regime'].unique())}\n")


def _n_valid(df, regime, arch, alpha, col):
    """Rows with a usable (non-NaN) value in `col`, or 0 if the frame/column is absent."""
    if df.empty or col not in df.columns:
        return 0
    m = (df["regime"] == regime) & (df["arch"] == arch) & (df["alpha"] == alpha)
    return int(pd.to_numeric(df.loc[m, col], errors="coerce").notna().sum())


rows = []
for r in RUNS:
    rows.append({
        "arch": r["arch"], "regime": r["regime"], "alpha": r["alpha"],
        "eval": _n_valid(EVAL, r["regime"], r["arch"], r["alpha"], "acc"),
        "gibbs": _n_valid(GIBBS, r["regime"], r["arch"], r["alpha"], "gibbs_clean_acc"),
        "dir": r["dir"].name,
    })
COVERAGE = pd.DataFrame(rows).sort_values(["arch", "regime", "alpha"]).reset_index(drop=True)
print("Coverage (effective seed counts, '.' = CSV absent):")
print(COVERAGE.to_string(index=False, formatters={
    "eval": lambda v: "." if v == 0 else str(v),
    "gibbs": lambda v: "." if v == 0 else str(v)}))

_gaps = COVERAGE[(COVERAGE["eval"] < 5) | (COVERAGE["gibbs"] < 5)]
print(f"\n{len(_gaps)} incomplete cell(s) — these show as gaps in the figures below:")
for _, g in _gaps.iterrows():
    print(f"  {g['regime']:>3} {g['arch']} a={g['alpha']:<5} eval={g['eval']} gibbs={g['gibbs']}")

---
## §0.2  Configuration

`BOND_DIM` is the Part-2 commitment. Everything downstream — figure directories included —
re-keys off it, so switching to 10 or 40 needs no other edit.

In [ ]:
import matplotlib.pyplot as plt

# ---- the Part-2 commitment -------------------------------------------------
BOND_DIM = 20
ARCH = f"d3r{BOND_DIM}c64"
# ---------------------------------------------------------------------------

EPS_LIST = [0.1, 0.2, 0.3]     # PGD budgets present in the CSVs
EPS_MAIN = 0.2                 # headline budget for single-epsilon figures
PURIFY_RADIUS = 0.2            # fixed absolute defense radius
PURIFY_RADII = ["0.2"]            # fixed absolute defense radius
GIBBS_KS = [1, 3, 6]           # Gibbs sweep counts present in the CSVs
# Headline Gibbs strength. NOTE: k=5 is NOT in this data — analysis/gibbs.py was run with
# --sweeps 1,3,6, so 6 is the nearest available. To get a real k=5, re-run
#   python analysis/gibbs.py <sweep_dir> --sweeps 1,3,5 --n-eval 250
# for every run dir, then set GIBBS_K = 5 (GIBBS_KS is auto-checked against the CSVs below).
GIBBS_K = 6
DET_PCT = "10pct"              # headline detection threshold (10th pct of clean log p(x))
PERCENTILES = [1, 5, 10, 20]   # detection thresholds present in the CSVs
GIBBS_N_EVAL = 250             # Gibbs subsample size — for captions

NAT_ALPHAS = [0.0, 0.01, 0.1, 0.2, 0.5, 1.0]
BOND_DIMS = sorted(EVAL["bond_dim"].unique())

# Canonical model order: the nat alpha ladder, then the adversarially-trained models.
MODEL_ORDER = [("nat", a) for a in NAT_ALPHAS] + [("at", 0.0), ("at", 0.01)]
# Models used where a 7-way comparison would be unreadable.
HEADLINE_MODELS = [("nat", 0.0), ("nat", 0.01), ("nat", 0.5), ("nat", 1.0), ("at", 0.0)]

_ALPHA_STR = {0.0: "0", 0.01: "0.01", 0.1: "0.1", 0.2: "0.2", 0.5: "0.5", 1.0: "1"}


def model_label(regime, alpha, tex=True):
    a = _ALPHA_STR[alpha]
    if not tex:
        return f"a={a}" if regime == "nat" else f"AT a={a}"
    return rf"$\alpha{{=}}{a}$" if regime == "nat" else rf"AT ($\alpha{{=}}{a}$)"


# Colours: the nat ladder walks viridis in alpha order; AT models get warm accents.
MODEL_COLOR = {("nat", a): tuple(c) for a, c in
               zip(NAT_ALPHAS, plt.cm.viridis(np.linspace(0.05, 0.88, len(NAT_ALPHAS))))}
MODEL_COLOR[("at", 0.0)] = "#D32F2F"
MODEL_COLOR[("at", 0.01)] = "#7B1FA2"

BOND_COLOR = dict(zip(BOND_DIMS, ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"]))
EPS_COLOR = {0.1: "steelblue", 0.2: "darkorange", 0.3: "seagreen"}

print(f"Part 2 arch = {ARCH}  (bond dims available: {BOND_DIMS})")
print(f"eps={EPS_LIST} main={EPS_MAIN}  radii={PURIFY_RADII} main={PURIFY_RADIUS}  "
      f"gibbs k={GIBBS_KS} main={GIBBS_K}")
print(f"figures -> {FIG_ROOT}")


def _suffixes(df, prefix, part=-1):
    """Distinct trailing path components of the `prefix...` columns actually in the CSVs."""
    return sorted({c.split("/")[part] for c in df.columns if c.startswith(prefix)})


# Guard the headline constants against the data, so a bad default fails loudly here rather
# than silently emitting empty lines and blank bars further down.
_have_eps = _suffixes(EVAL, "rob/", part=1)
_have_rad = _suffixes(EVAL, "uq_purify_acc/")
_have_k = _suffixes(GIBBS, "gibbs_purify_acc/")
print(f"\nin the CSVs: eps={_have_eps}  purify radii={_have_rad}  gibbs k={_have_k}")
for name, val, have in (("EPS_MAIN", str(EPS_MAIN), _have_eps),
                        ("PURIFY_RADIUS", str(PURIFY_RADIUS), _have_rad),
                        ("GIBBS_K", str(GIBBS_K), _have_k)):
    flag = "OK " if val in have else "!! "
    print(f"  {flag}{name} = {val}" + ("" if val in have else f"  <-- NOT PRESENT, pick from {have}"))

---
## §0.3  Shared helpers

Selection, aggregation, figure paths, and the two curve builders used repeatedly below.

**`accept_curve`** deserves a note — it is the basis of the accept-rate figures (§1.3, §2.5):

- **x** = ratio of passed samples = $1 - $ `uq_detection/{q}pct/{eps}`
- **y** = accuracy on passed samples = $1 - $ `uq_det_err_passed/{q}pct/{eps}`

Both are measured **on the adversarial set**: `uq_detection` is the fraction of *adversarial*
inputs whose $\log p(x)$ falls below the threshold $\tau_q$ (itself the $q$-th percentile of
*clean* $\log p(x)$), and `uq_det_err_passed` is the misclassification rate among the survivors.
The passed ratio is therefore **not** simply $1 - q/100$, and the curve traces a genuine
trade-off rather than a reparametrisation of $q$. The $q=0$ point (accept everything) is
anchored at $(1.0,\ $`uq_adv_acc/{eps}`$)$.

In [ ]:
def sel(df, regime=None, arch=None, alpha=None, bond_dim=None):
    """Row subset of EVAL/GIBBS matching the given registry keys."""
    if df.empty:
        return df
    m = pd.Series(True, index=df.index)
    for key, val in (("regime", regime), ("arch", arch),
                     ("alpha", alpha), ("bond_dim", bond_dim)):
        if val is not None:
            m &= df[key] == val
    return df[m]


def ms(df, col, flip=False):
    """(mean, std) over seeds, NaN-dropping. (nan, nan) if the frame/column is unusable."""
    if df is None or df.empty or col not in df.columns:
        return float("nan"), float("nan")
    v = pd.to_numeric(df[col], errors="coerce").dropna()
    if len(v) == 0:
        return float("nan"), float("nan")
    if flip:
        v = 1.0 - v
    return float(v.mean()), float(v.std() if len(v) > 1 else 0.0)


def fig_path(group, name, arch=None):
    """figures/mnist_r12/{arch or 'compare'}/{group}/{name}, parents created."""
    d = FIG_ROOT / (arch if arch else "compare") / group
    d.mkdir(parents=True, exist_ok=True)
    return d / name


def save(fig, path):
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"Saved {path.relative_to(PROJECT_ROOT)}")


def setup_alpha_axis(ax):
    """Symlog alpha axis ticked at the six trained alphas."""
    ax.set_xscale("symlog", linthresh=5e-3)
    ax.set_xlim(-2e-4, 1.3)
    ax.set_xticks(NAT_ALPHAS)
    ax.set_xticklabels([_ALPHA_STR[a] for a in NAT_ALPHAS], rotation=45, ha="right")
    ax.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    ax.set_xlabel(r"$\alpha$")


def accept_curve(df, eps):
    """(q, passed_ratio, acc_on_passed, xerr, yerr) over q in [0] + PERCENTILES.

    x = 1 - uq_detection/{q}pct/{eps}       fraction of adversarial inputs NOT flagged
    y = 1 - uq_det_err_passed/{q}pct/{eps}  accuracy among those that passed
    q = 0 (accept everything) anchors at (1.0, uq_adv_acc/{eps}).
    Points stay in q order, which is monotone decreasing in x.
    """
    qs, xs, ys, xe, ye = [], [], [], [], []
    m, s = ms(df, f"uq_adv_acc/{eps}")
    if not np.isnan(m):
        qs.append(0); xs.append(1.0); ys.append(m); xe.append(0.0); ye.append(s)
    for q in PERCENTILES:
        cd, cp = f"uq_detection/{q}pct/{eps}", f"uq_det_err_passed/{q}pct/{eps}"
        xm, xs_ = ms(df, cd, flip=True)
        ym, ys_ = ms(df, cp, flip=True)
        if np.isnan(xm) or np.isnan(ym):
            continue
        qs.append(q); xs.append(xm); ys.append(ym); xe.append(xs_); ye.append(ys_)
    return (np.array(qs), np.array(xs), np.array(ys),
            np.nan_to_num(np.array(xe)), np.nan_to_num(np.array(ye)))


def gibbs_curve(df, eps=None):
    """(k, mean, std) over k in [0] + GIBBS_KS. k=0 is the paired undefended anchor.

    eps=None gives the clean-input curve (what purification costs when nothing attacked).
    Both the anchor and the purified points come from the same 250-sample subsample, so
    the k=0 -> k>0 lift is paired.
    """
    ks, mm, ss = [], [], []
    for k in [0] + GIBBS_KS:
        if eps is None:
            col = "gibbs_clean_acc" if k == 0 else f"gibbs_clean_purify_acc/{k}"
        else:
            col = f"gibbs_adv_acc/{eps}" if k == 0 else f"gibbs_purify_acc/{eps}/{k}"
        m, s = ms(df, col)
        if np.isnan(m):
            continue
        ks.append(k); mm.append(m); ss.append(s)
    return np.array(ks), np.array(mm), np.nan_to_num(np.array(ss))


def errline(ax, x, y, s, color, label=None, ls="-", marker="o"):
    """Line + 1-std band, skipping empty series."""
    if len(x) == 0:
        return
    ax.plot(x, y, color=color, ls=ls, lw=1.8, marker=marker, ms=5, label=label)
    ax.fill_between(x, np.asarray(y) - s, np.asarray(y) + s, color=color, alpha=0.15)


print("helpers ready")

---
## §0.4  Reproduction commands

The trained checkpoints are **not local** — `run_path` in every consolidated CSV points at
`/mathqi/mnissen/bm4tc/outputs/...`. This notebook never loads a model; it reads CSVs and the
sampling PNGs that were transferred alongside them. The cell below prints the commands that
produced this data, for the record.

In [ ]:
print("# Training (per bond dim) — 3-stage alpha-HPO pipeline, run from the project root:")
for R in BOND_DIMS:
    a = f"{DATASET}/nat/{EMBEDDING}/d3r{R}c64"
    print(f"python -m experiments.train --multirun +experiments={a}/hpo_a0")
    print(f"#   then hpo_pretrained_a{{001,01,02,05,1}} warm-started from the a0 checkpoint,")
    print(f"#   tools/fill_hpo.py, then seed_sweep_a{{0,001,01,02,05,1}}")
print("\n# Adversarial training warm-starts from the matching a0 checkpoint:")
print("python tools/patch_checkpoint.py outputs/.../seed_sweep_a0_DDMM")
for R in BOND_DIMS:
    print(f"python -m experiments.train --multirun "
          f"+experiments={DATASET}/at/{EMBEDDING}/d3r{R}c64/seed_sweep")

print("\n# Post-hoc analysis — one pass per run dir:")
print("python analysis/sweep.py <sweep_dir> --viz          # -> evaluation_data.csv (+ sampling PNGs)")
print(f"python analysis/gibbs.py <sweep_dir> --sweeps {','.join(map(str, GIBBS_KS))} "
      f"--n-eval {GIBBS_N_EVAL}   # -> gibbs_data.csv")

---
---
# Part 1 — Bond-dimension comparison

All figures here span **every** analysed bond dimension and land in
`figures/mnist_r12/compare/`. The question this part answers is *which bond dimension to
commit to* in Part 2.

## §1.1  Every headline metric vs bond dimension

Four panels at $\varepsilon = $ `EPS_MAIN`, one line per model, mean $\pm$ 1 std over seeds:

| panel | column |
|---|---|
| Clean | `acc` |
| Rob. (undefended) | `rob/{eps}` |
| Purif. (lk.) | `uq_purify_acc/{eps}/{PURIFY_RADIUS}` |
| Purif. (Gibbs) | `gibbs_purify_acc/{eps}/{GIBBS_K}` |

The Gibbs panel is on a 250-sample subsample — its vertical scale carries ~±3% sampling
noise that the other three panels do not. Missing points are genuine coverage gaps
(see the §0.1 table), not plotting failures.

In [ ]:
PANELS = [
    ("Clean",                             EVAL,  "acc",                                        False),
    (r"Rob.\ (undefended)" if USE_LATEX else "Rob. (undefended)",
                                          EVAL,  f"rob/{EPS_MAIN}",                            False),
    (rf"Purif.\ (lk., $r{{=}}{PURIFY_RADIUS}$)" if USE_LATEX else "Purif. (lk.)",
                                          EVAL,  f"uq_purify_acc/{EPS_MAIN}/{PURIFY_RADIUS}",  False),
    (rf"Purif.\ (Gibbs, $k{{=}}{GIBBS_K}$)" if USE_LATEX else "Purif. (Gibbs)",
                                          GIBBS, f"gibbs_purify_acc/{EPS_MAIN}/{GIBBS_K}",      True),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
for ax, (title, src, col, is_gibbs) in zip(axes.ravel(), PANELS):
    for regime, alpha in MODEL_ORDER:
        pts = [(bd, *ms(sel(src, regime=regime, bond_dim=bd, alpha=alpha), col))
               for bd in BOND_DIMS]
        pts = [(b, m, s) for b, m, s in pts if not np.isnan(m)]
        if not pts:
            continue
        b, m, s = map(np.array, zip(*pts))
        errline(ax, b, m, s, MODEL_COLOR[(regime, alpha)], model_label(regime, alpha))
    ax.set_title(title)
    ax.set_xscale("log"); ax.set_xticks(BOND_DIMS)
    ax.set_xticklabels([str(b) for b in BOND_DIMS])
    ax.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    ax.set_ylim(0, 1.02); ax.grid(True, alpha=0.3)
    if is_gibbs:
        ax.text(0.03, 0.04, rf"$n_{{\mathrm{{eval}}}}={GIBBS_N_EVAL}$", transform=ax.transAxes,
                fontsize=10, color="dimgray")
for ax in axes[1, :]:
    ax.set_xlabel("bond dimension $r$")
for ax in axes[:, 0]:
    ax.set_ylabel("Accuracy")
axes[0, 0].legend(ncol=2, fontsize=9, loc="lower right")
fig.suptitle(rf"MNIST 12$\times$12 — metrics vs bond dimension ($\varepsilon={EPS_MAIN}$)",
             fontsize=15)
fig.tight_layout()
save(fig, fig_path("bonddim", "metric_vs_bonddim.pdf"))

## §1.2  Alpha curve, one line per bond dimension

The same alpha ladder drawn once per bond dimension — clean accuracy (solid) against
undefended robustness (dashed) at $\varepsilon = $ `EPS_MAIN`. This is the figure to read when
choosing `BOND_DIM`: it shows whether the clean/robust trade-off across $\alpha$ has the same
shape at every bond dimension, or whether capacity changes the story.

In [ ]:
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
for bd in BOND_DIMS:
    c = BOND_COLOR[bd]
    for ax, col in ((ax_l, "acc"), (ax_r, f"rob/{EPS_MAIN}")):
        pts = [(a, *ms(sel(EVAL, regime="nat", bond_dim=bd, alpha=a), col)) for a in NAT_ALPHAS]
        pts = [(a, m, s) for a, m, s in pts if not np.isnan(m)]
        if not pts:
            continue
        a, m, s = map(np.array, zip(*pts))
        errline(ax, a, m, s, c, rf"$r{{=}}{bd}$")
    # AT reference at this bond dim (alpha-independent): horizontal dotted line.
    m, _ = ms(sel(EVAL, regime="at", bond_dim=bd, alpha=0.0), f"rob/{EPS_MAIN}")
    if not np.isnan(m):
        ax_r.axhline(m, color=c, ls=":", lw=1.4, alpha=0.8)

for ax, title in ((ax_l, "Clean accuracy"),
                  (ax_r, rf"Robust accuracy ($\varepsilon={EPS_MAIN}$)")):
    setup_alpha_axis(ax); ax.set_title(title)
    ax.set_ylim(0, 1.02); ax.grid(True, alpha=0.3)
ax_l.set_ylabel("Accuracy")
ax_l.legend(loc="lower left")
ax_r.plot([], [], color="gray", ls=":", lw=1.4, label="AT (same $r$)")
ax_r.legend(loc="upper left")
fig.tight_layout()
save(fig, fig_path("bonddim", "alpha_curve_by_bonddim.pdf"))

## §1.3  Accuracy on passed samples vs ratio of passed samples

**x** = ratio of passed samples $= 1 - $ detection rate · **y** = accuracy among those passed.
One marker per rejection threshold $q \in \{0, 1, 5, 10, 20\}\%$, joined in $q$ order
(increasing $q$ moves left: more adversarial inputs are flagged, fewer pass). Error bars are
1 std over seeds in **both** axes. Rows are models, columns are attack budgets, one line per
bond dimension; only the two extreme operating points are labelled, to keep the panels legible.

Read it as an operating-point curve: the top-right corner is ideal — keep everything and still
be accurate. A curve that rises steeply as it moves left means the detector is buying real
accuracy per sample discarded; a flat curve means it is discarding samples for nothing.

**y is shared per row, not globally.** The models sit at very different accuracy levels
(AT $\approx 0.85$ against $\alpha=0$ at $\approx 0.00$ for $\varepsilon=0.3$), so one shared
scale would flatten most panels into a line.

> **What these curves show.** Detection helps only where the attack has not already won.
> The one clearly useful case is $\alpha=0$ at $\varepsilon=0.1$, where rejecting the
> lowest-likelihood inputs is worth real accuracy and the benefit **grows with bond dimension**
> ($r{=}40$: $0.61 \to 0.74$ as the passed ratio falls $1.0 \to 0.41$; $r{=}10$ only
> $0.33 \to 0.48$). Everywhere else the curves are close to flat, and by $\varepsilon=0.3$ the
> $\alpha=0$ model is at $\approx 0$ accuracy however much is discarded — nothing can be
> recovered by filtering when every surviving sample is already misclassified. For AT the curve
> is flat *because it starts high*: it needs no filtering. Flatness is a result here, not a
> plotting artefact — compare the purification columns in §1.5.

In [ ]:
ACC_MODELS = [("nat", 0.0), ("nat", 0.01), ("at", 0.0)]


def label_endpoints(ax, q, x, y):
    """Label only the two extreme operating points; x is monotone decreasing in q."""
    if len(q) < 2:
        return
    for idx, ha in ((0, "left"), (len(q) - 1, "right")):
        ax.annotate(rf"$q{{=}}{int(q[idx])}{PCT}$", (x[idx], y[idx]),
                    textcoords="offset points", xytext=(0, 8), ha=ha,
                    fontsize=8, color="dimgray")


# Rows = model, columns = epsilon. y is shared per ROW: the models sit at wildly different
# accuracy levels (AT ~0.85, alpha=0 ~0.00 at eps=0.3), so a single shared scale would
# flatten most panels into a line.
fig, axes = plt.subplots(len(ACC_MODELS), len(EPS_LIST),
                         figsize=(4.4 * len(EPS_LIST), 3.5 * len(ACC_MODELS)),
                         sharey="row", squeeze=False)
for row, (regime, alpha) in enumerate(ACC_MODELS):
    for col, eps in enumerate(EPS_LIST):
        ax = axes[row][col]
        for i, bd in enumerate(BOND_DIMS):
            q, x, y, xe, ye = accept_curve(sel(EVAL, regime=regime, bond_dim=bd, alpha=alpha),
                                           eps)
            if len(x) == 0:
                continue
            ax.errorbar(x, y, xerr=xe, yerr=ye, color=BOND_COLOR[bd], lw=1.8, marker="o",
                        ms=5, capsize=2, elinewidth=0.9, label=rf"$r{{=}}{bd}$")
            if i == 0:
                label_endpoints(ax, q, x, y)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(bottom=0)
        ax.margins(y=0.22)
        if row == 0:
            ax.set_title(rf"$\varepsilon={eps}$")
        if row == len(ACC_MODELS) - 1:
            ax.set_xlabel("Ratio of passed samples")
    axes[row][0].set_ylabel(model_label(regime, alpha) + "\nAcc. on passed")
axes[0][0].legend(fontsize=9, loc="lower left")
fig.suptitle("Detection operating curve — accuracy on passed vs ratio passed", fontsize=15)
fig.tight_layout()
save(fig, fig_path("bonddim", "accept_curve_by_bonddim.pdf"))

## §1.4  Training losses vs bond dimension

Discriminative NLL (`dis_loss`, left axis) and generative NLL (`gen_loss`, right axis) against
bond dimension, one line per $\alpha$. The two axes differ by orders of magnitude — the
generative NLL is a 144-dimensional log-density, the discriminative one a 10-class log-loss.

In [ ]:
fig, ax_l = plt.subplots(figsize=(7, 4.4))
ax_r = ax_l.twinx()
for alpha in NAT_ALPHAS:
    c = MODEL_COLOR[("nat", alpha)]
    for ax, col, ls in ((ax_l, "dis_loss", "-"), (ax_r, "gen_loss", "--")):
        pts = [(bd, *ms(sel(EVAL, regime="nat", bond_dim=bd, alpha=alpha), col))
               for bd in BOND_DIMS]
        pts = [(b, m, s) for b, m, s in pts if not np.isnan(m)]
        if not pts:
            continue
        b, m, s = map(np.array, zip(*pts))
        lab = model_label("nat", alpha) if ax is ax_l else None
        errline(ax, b, m, s, c, lab, ls=ls, marker="o" if ax is ax_l else "s")

ax_l.set_xscale("log"); ax_l.set_xticks(BOND_DIMS)
ax_l.set_xticklabels([str(b) for b in BOND_DIMS])
ax_l.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
ax_l.set_xlabel("bond dimension $r$")
ax_l.set_ylabel("Dis. NLL (solid)", color="darkred")
ax_r.set_ylabel("Gen. NLL (dashed)", color="steelblue")
ax_l.grid(True, alpha=0.3)
ax_l.legend(ncol=2, fontsize=9, loc="best")
fig.tight_layout()
save(fig, fig_path("bonddim", "nll_vs_bonddim.pdf"))

## §1.5  Cross-bond-dimension summary table

Rows are (bond dimension × model), columns the five headline metrics at
$\varepsilon = $ `EPS_MAIN`. Best per column in **bold**. Written as booktabs `.tex`
(needs `\usepackage{booktabs,multirow}`) and an aligned `.txt` preview.

In [ ]:
def tex_cell(m, s, bold=False, signed=False):
    if np.isnan(m):
        return "---"
    num = f"{m:+.3f}" if signed else f"{m:.3f}"
    body = f"{num} \\pm {s:.3f}" if (not np.isnan(s) and s != 0.0) else num
    return f"$\\mathbf{{{body}}}$" if bold else f"${body}$"


def txt_cell(m, s, best=False, signed=False):
    if np.isnan(m):
        return "---"
    num = f"{m:+.3f}" if signed else f"{m:.3f}"
    return (f"{num}±{s:.3f}" if (not np.isnan(s) and s != 0.0) else num) + ("*" if best else "")


def strip_tex(t):
    """Crude LaTeX -> plain, for the .txt previews."""
    for a, b in ((r"\ ", " "), (r"\%", "%"), ("$", ""), ("{=}", "="), (r"\alpha", "a"),
                 (r"\varepsilon", "eps"), (r"\mathbf", ""), (r"\times", "x"),
                 ("{", ""), ("}", "")):
        t = t.replace(a, b)
    return t


def best_index(col_means):
    """Index of the largest non-NaN mean (all metrics here are higher-is-better)."""
    valid = [(v, i) for i, v in enumerate(col_means) if not np.isnan(v)]
    return max(valid)[1] if valid else -1


# (header, source frame, column, flip)
SUMMARY_METRICS = [
    ("Clean",                                            EVAL,  "acc", False),
    (r"Rob.",                                            EVAL,  f"rob/{EPS_MAIN}", False),
    (rf"Purif.\ (lk., $r{{=}}{PURIFY_RADIUS}$)",         EVAL,  f"uq_purify_acc/{EPS_MAIN}/{PURIFY_RADIUS}", False),
    (rf"Purif.\ (Gibbs, $k{{=}}{GIBBS_K}$)",             GIBBS, f"gibbs_purify_acc/{EPS_MAIN}/{GIBBS_K}", False),
    (rf"Acc.\ passed ($q{{=}}10\%$)",                    EVAL,  f"uq_det_err_passed/{DET_PCT}/{EPS_MAIN}", True),
]

body = []          # [(row_label_tex, row_label_txt, bond_dim, [(mean, std), ...])]
for bd in BOND_DIMS:
    for regime, alpha in MODEL_ORDER:
        cells = [ms(sel(src, regime=regime, bond_dim=bd, alpha=alpha), col, flip)
                 for _, src, col, flip in SUMMARY_METRICS]
        if all(np.isnan(m) for m, _ in cells):
            continue
        body.append((model_label(regime, alpha), model_label(regime, alpha, tex=False),
                     bd, cells))

best = [best_index([r[3][j][0] for r in body]) for j in range(len(SUMMARY_METRICS))]

L = [r"\begin{tabular}{@{}ll" + "r" * len(SUMMARY_METRICS) + r"@{}}", r"\toprule",
     r"$r$ & Model & " + " & ".join(h for h, *_ in SUMMARY_METRICS) + r" \\", r"\midrule"]
prev_bd = None
for i, (lab, _, bd, cells) in enumerate(body):
    if bd != prev_bd:
        if prev_bd is not None:
            L.append(r"\cmidrule(lr){1-%d}" % (2 + len(SUMMARY_METRICS)))
        n = sum(1 for r in body if r[2] == bd)
        first = f"\\multirow{{{n}}}{{*}}{{{bd}}}"
        prev_bd = bd
    else:
        first = ""
    L.append(f"{first} & {lab} & " +
             " & ".join(tex_cell(m, s, bold=(best[j] == i)) for j, (m, s) in enumerate(cells)) +
             r" \\")
L += [r"\bottomrule", r"\end{tabular}"]

W = 20
hdr = f"{'r':<5}{'Model':<12}" + "".join(f"{strip_tex(h):<{W}}" for h, *_ in SUMMARY_METRICS)
P = [f"MNIST 12x12 — summary at eps={EPS_MAIN} "
     f"(lk. radius {PURIFY_RADIUS}, Gibbs k={GIBBS_K} on n_eval={GIBBS_N_EVAL}); * = best",
     hdr, "-" * len(hdr)]
prev_bd = None
for i, (_, lab, bd, cells) in enumerate(body):
    if bd != prev_bd and prev_bd is not None:
        P.append("-" * len(hdr))
    P.append(f"{(str(bd) if bd != prev_bd else ''):<5}{lab:<12}" +
             "".join(f"{txt_cell(m, s, best=(best[j] == i)):<{W}}"
                     for j, (m, s) in enumerate(cells)))
    prev_bd = bd

fig_path("tables", ".keep").parent  # ensure dir
(fig_path("tables", "bonddim_summary.tex")).write_text("\n".join(L) + "\n")
(fig_path("tables", "bonddim_summary.txt")).write_text("\n".join(P) + "\n")
print(f"Saved {fig_path('tables', 'bonddim_summary.tex').relative_to(PROJECT_ROOT)} "
      f"and .txt\n")
print("\n".join(P))

---
---
# Part 2 — Committed bond dimension

Everything below is for **`BOND_DIM`** (set in §0.2, default 20) and lands in
`figures/mnist_r12/{ARCH}/`. Change `BOND_DIM` and re-run from §0.2 to regenerate the whole
part at another bond dimension.

In [ ]:
print(f"Part 2 — arch = {ARCH}  (bond dim {BOND_DIM})")
print(f"figures -> {(FIG_ROOT / ARCH).relative_to(PROJECT_ROOT)}")
_present = [(r, a) for r, a in MODEL_ORDER
            if not sel(EVAL, regime=r, arch=ARCH, alpha=a).empty]
_gibbs_present = [(r, a) for r, a in MODEL_ORDER
                  if not sel(GIBBS, regime=r, arch=ARCH, alpha=a).empty]
print(f"models with eval  : {[model_label(r, a, tex=False) for r, a in _present]}")
print(f"models with gibbs : {[model_label(r, a, tex=False) for r, a in _gibbs_present]}")

## §2.1  Sampling — per-class mean digit

Assembled from the **pre-rendered** `mnist_samples.png` that `analysis/sweep.py --viz` wrote
next to each CSV (preference: `mean/` → `mean64/` → run root → `single/`). Nothing is sampled
here — the checkpoints are on mathqi.

Each tile is a 2×5 grid of per-class mean sampled digits (0–9) from the best run of that seed
sweep. Legibility should improve as $\alpha$ rises: at $\alpha=0$ the model is a pure
classifier and its "samples" carry no digit structure.

In [ ]:
from PIL import Image

SAMPLE_SUBDIRS = ["mean", "mean64", ".", "single"]


def sample_png(regime, alpha, arch=ARCH):
    for r in RUNS:
        if (r["regime"], r["arch"], r["alpha"]) != (regime, arch, alpha):
            continue
        for sub in SAMPLE_SUBDIRS:
            p = (r["dir"] / sub / "mnist_samples.png") if sub != "." else (r["dir"] / "mnist_samples.png")
            if p.exists():
                return p
    return None


cols = [(r, a) for r, a in MODEL_ORDER if sample_png(r, a) is not None]
fig, axes = plt.subplots(1, max(len(cols), 1), figsize=(2.3 * max(len(cols), 1), 3.0),
                         squeeze=False)
for ax, (regime, alpha) in zip(axes[0], cols):
    ax.imshow(np.array(Image.open(sample_png(regime, alpha)).convert("RGB")))
    ax.set_title(model_label(regime, alpha), fontsize=12)
    ax.axis("off")
for ax in axes[0][len(cols):]:
    ax.axis("off")
missing = [model_label(r, a, tex=False) for r, a in MODEL_ORDER if sample_png(r, a) is None]
if missing:
    print(f"no sampling PNG for: {missing}")
fig.subplots_adjust(wspace=0.03)
save(fig, fig_path("sampling", "mnist_samples.pdf", arch=ARCH))

## §2.2  Alpha curve — accuracy

Every defense against $\alpha$ at $\varepsilon = $ `EPS_MAIN`, mean $\pm$ 1 std over seeds:
clean accuracy, undefended robustness, likelihood purification at **both** radii, and Gibbs
purification at $k = $ `GIBBS_K`. The adversarially-trained model is $\alpha$-independent and
is drawn as a horizontal reference band (its undefended robustness).

The Gibbs line sits on a 250-sample subsample; the others use the full test split.

In [ ]:
CURVES = [
    ("Clean",                                    EVAL,  "acc",                                       "#455A64", "-"),
    (r"Rob.\ (undefended)" if USE_LATEX else "Rob. (undefended)",
                                                 EVAL,  f"rob/{EPS_MAIN}",                           "#FF9800", "--"),
    (r"Purif.\ (lk., $r{=}0.2$)" if USE_LATEX else "Purif. (lk., r=0.2)",
                                                 EVAL,  f"uq_purify_acc/{EPS_MAIN}/0.2",             "#4CAF50", "-."),
    (rf"Purif.\ (Gibbs, $k{{=}}{GIBBS_K}$)" if USE_LATEX else f"Purif. (Gibbs, k={GIBBS_K})",
                                                 GIBBS, f"gibbs_purify_acc/{EPS_MAIN}/{GIBBS_K}",    "#7B1FA2", ":"),
]

fig, ax = plt.subplots(figsize=(7.5, 4.8))
for label, src, col, color, ls in CURVES:
    pts = [(a, *ms(sel(src, regime="nat", arch=ARCH, alpha=a), col)) for a in NAT_ALPHAS]
    pts = [(a, m, s) for a, m, s in pts if not np.isnan(m)]
    if not pts:
        print(f"skip (no data): {strip_tex(label)}")
        continue
    a, m, s = map(np.array, zip(*pts))
    errline(ax, a, m, s, color, label, ls=ls)

m, s = ms(sel(EVAL, regime="at", arch=ARCH, alpha=0.0), f"rob/{EPS_MAIN}")
if not np.isnan(m):
    ax.axhline(m, color="#D32F2F", ls="--", lw=1.5,
               label=r"AT rob.\ (undefended)" if USE_LATEX else "AT rob.")
    ax.axhspan(m - s, m + s, color="#D32F2F", alpha=0.12)

setup_alpha_axis(ax)
ax.set_ylabel("Accuracy"); ax.set_ylim(0, 1.02); ax.grid(True, alpha=0.3)
ax.set_title(rf"{ARCH} — accuracy vs $\alpha$ ($\varepsilon={EPS_MAIN}$)")
ax.legend(fontsize=10, loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
save(fig, fig_path("alpha", "alpha_curve_accuracy.pdf", arch=ARCH))

## §2.3  Alpha curve — NLL

Discriminative and generative NLL against $\alpha$ on separate axes. This is the trade-off the
mixed objective is interpolating: raising $\alpha$ buys density modelling at the cost of
class-conditional sharpness.

In [ ]:
fig, ax_l = plt.subplots(figsize=(7, 4.4))
ax_r = ax_l.twinx()
for ax, col, color, ls, mk in ((ax_l, "dis_loss", "darkred", "-", "o"),
                               (ax_r, "gen_loss", "steelblue", "--", "s")):
    pts = [(a, *ms(sel(EVAL, regime="nat", arch=ARCH, alpha=a), col)) for a in NAT_ALPHAS]
    pts = [(a, m, s) for a, m, s in pts if not np.isnan(m)]
    if not pts:
        continue
    a, m, s = map(np.array, zip(*pts))
    errline(ax, a, m, s, color, col, ls=ls, marker=mk)

setup_alpha_axis(ax_l)
ax_l.set_ylabel("Dis. NLL (solid)", color="darkred")
ax_r.set_ylabel("Gen. NLL (dashed)", color="steelblue")
ax_l.tick_params(axis="y", labelcolor="darkred")
ax_r.tick_params(axis="y", labelcolor="steelblue")
ax_l.grid(True, alpha=0.3)
ax_l.set_title(rf"{ARCH} — NLL vs $\alpha$")
fig.tight_layout()
save(fig, fig_path("alpha", "alpha_curve_nll.pdf", arch=ARCH))

## §2.4  Defense comparison bar plot

The spirals-paper figure, one standalone file per attack budget
($\varepsilon$ in the filename). x = model, bar groups = defense, mean $\pm$ 1 std over seeds:

- **Clean** — clean accuracy reference (`acc`, $\varepsilon$-independent)
- **Rob.** — undefended adversarial accuracy (`rob/{eps}`)
- **Purif. (lk.)** — likelihood purification at radius `PURIFY_RADIUS`
- **Purif. (Gibbs)** — Gibbs purification at $k = $ `GIBBS_K` *(250-sample subsample)*
- **Accept** — accuracy on inputs the detector does not flag,
  $1-$`uq_det_err_passed/10pct/{eps}`

Note **Accept** is an accuracy *conditional on passing*, so it is not directly comparable to
the unconditional bars beside it — §2.5 shows what fraction of samples that conditioning keeps.

In [ ]:
DEFENSES = [
    ("Clean",                                    "#9E9E9E", EVAL,  "acc",                                       False),
    ("Rob.",                                     "#FF9800", EVAL,  "rob/{eps}",                                 False),
    (r"Purif.\ (lk.)" if USE_LATEX else "Purif. (lk.)",
                                                 "#4CAF50", EVAL,  f"uq_purify_acc/{{eps}}/{PURIFY_RADIUS}",    False),
    (rf"Purif.\ (Gibbs $k{{=}}{GIBBS_K}$)" if USE_LATEX else f"Purif. (Gibbs k={GIBBS_K})",
                                                 "#7B1FA2", GIBBS, f"gibbs_purify_acc/{{eps}}/{GIBBS_K}",       False),
    (rf"Accept ($q{{=}}10{PCT}$)",               "#2196F3", EVAL,  f"uq_det_err_passed/{DET_PCT}/{{eps}}",      True),
]

models = [(r, a) for r, a in HEADLINE_MODELS if not sel(EVAL, regime=r, arch=ARCH, alpha=a).empty]
x, n_d = np.arange(len(models)), len(DEFENSES)
width = 0.8 / n_d

for eps in EPS_LIST:
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    for j, (label, color, src, tmpl, flip) in enumerate(DEFENSES):
        stats = [ms(sel(src, regime=r, arch=ARCH, alpha=a), tmpl.format(eps=eps), flip)
                 for r, a in models]
        m, s = zip(*stats)
        ax.bar(x + (j - (n_d - 1) / 2) * width, np.nan_to_num(m), width,
               yerr=np.nan_to_num(s), label=label, color=color, capsize=2, alpha=0.9,
               error_kw={"elinewidth": 1})
    ax.set_xticks(x)
    ax.set_xticklabels([model_label(r, a) for r, a in models])
    ax.set_ylabel("Accuracy"); ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3, ls="--"); ax.set_axisbelow(True)
    ax.set_title(rf"{ARCH} — defenses at $\varepsilon={eps}$")
    ax.legend(fontsize=9, ncol=2, loc="upper right")
    fig.tight_layout()
    save(fig, fig_path("robustness", f"defense_comparison_eps{eps}.pdf", arch=ARCH))

## §2.5  Detection — what rejection actually buys

Two views of the same likelihood detector at the committed bond dimension.

**§2.5a — operating curve.** Accuracy on passed samples against the ratio of passed samples
(the §1.3 axes), one line per model, one panel per $\varepsilon$. This answers *what does
rejecting $q\%$ buy* — the further a curve sits toward the top-right, the better the detector
separates adversarial from clean inputs for that model. Here $y$ *is* shared across panels:
the spread between models is the point, and it survives the common scale.

As in §1.3, vertical separation between models (which model you trained) dominates movement
along any single curve (how much you reject) — the choice of $\alpha$ or AT matters far more
than the choice of $q$.

**§2.5b — threshold view.** The same quantities against $q$ itself, one figure per model:
accuracy on accepted on the left axis (solid, ○) and detection rate on the right (dashed, ■),
one colour per $\varepsilon$. This answers *which $q$ to pick*.

In [ ]:
models = [(r, a) for r, a in MODEL_ORDER if not sel(EVAL, regime=r, arch=ARCH, alpha=a).empty]

fig, axes = plt.subplots(1, len(EPS_LIST), figsize=(4.9 * len(EPS_LIST), 4.4),
                         sharey=True, squeeze=False)
for ax, eps in zip(axes[0], EPS_LIST):
    for i, (regime, alpha) in enumerate(models):
        q, x, y, xe, ye = accept_curve(sel(EVAL, regime=regime, arch=ARCH, alpha=alpha), eps)
        if len(x) == 0:
            continue
        ax.errorbar(x, y, xerr=xe, yerr=ye, color=MODEL_COLOR[(regime, alpha)], lw=1.7,
                    marker="o", ms=4, capsize=2, elinewidth=0.8,
                    label=model_label(regime, alpha))
        if i == 0:
            label_endpoints(ax, q, x, y)
    ax.set_title(rf"$\varepsilon={eps}$")
    ax.set_xlabel("Ratio of passed samples")
    ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.02)
axes[0][0].set_ylabel("Accuracy on passed samples")
axes[0][0].legend(fontsize=9, loc="lower left")
fig.suptitle(f"{ARCH} — detection operating curve", fontsize=15)
fig.tight_layout()
save(fig, fig_path("detection", "accept_acc_vs_passed_ratio.pdf", arch=ARCH))

In [ ]:
from matplotlib.lines import Line2D


def threshold_curve(df, eps, q0_col, tmpl, flip=False):
    """(q, mean, std) vs rejection percentile, with a q=0 anchor (None -> anchor at 0)."""
    qs, mm, ss = [], [], []
    if q0_col is None:
        qs.append(0.0); mm.append(0.0); ss.append(0.0)
    else:
        m, s = ms(df, q0_col)
        if not np.isnan(m):
            qs.append(0.0); mm.append(m); ss.append(s)
    for q in PERCENTILES:
        m, s = ms(df, tmpl.format(q=q, eps=eps), flip=flip)
        if np.isnan(m):
            continue
        qs.append(float(q)); mm.append(m); ss.append(s)
    return np.array(qs), np.array(mm), np.nan_to_num(np.array(ss))


for regime, alpha in [("nat", 0.0), ("nat", 0.01), ("nat", 1.0), ("at", 0.0)]:
    df = sel(EVAL, regime=regime, arch=ARCH, alpha=alpha)
    if df.empty:
        print(f"skip (no eval CSV): {model_label(regime, alpha, tex=False)}")
        continue
    fig, axl = plt.subplots(figsize=(6.2, 4.4))
    axr = axl.twinx()
    handles = []
    for eps in EPS_LIST:
        c = EPS_COLOR[eps]
        q, m, s = threshold_curve(df, eps, f"uq_adv_acc/{eps}",
                                  "uq_det_err_passed/{q}pct/{eps}", flip=True)
        if len(q):
            (ln,) = axl.plot(q, m, color=c, ls="-", lw=1.8, marker="o", ms=4,
                             label=rf"$\varepsilon{{=}}{eps}$")
            axl.fill_between(q, m - s, m + s, color=c, alpha=0.15)
            handles.append(ln)
        q, m, s = threshold_curve(df, eps, None, "uq_detection/{q}pct/{eps}")
        if len(q):
            axr.plot(q, m, color=c, ls="--", lw=1.6, marker="s", ms=4)
            axr.fill_between(q, m - s, m + s, color=c, alpha=0.10)

    axl.set_xlabel(rf"Rejection threshold $q$ (clean percentile, ${PCT}$)")
    axl.set_ylabel("Accuracy on accepted")
    axr.set_ylabel("Detection rate")
    axl.set_ylim(0, 1.05); axr.set_ylim(0, 1.05); axl.grid(True, alpha=0.3)
    leg1 = axl.legend(handles=handles, title=model_label(regime, alpha), loc="upper left")
    axl.add_artist(leg1)
    axl.legend(handles=[Line2D([], [], color="gray", ls="-", marker="o", ms=4,
                               label="Acc. accepted"),
                        Line2D([], [], color="gray", ls="--", marker="s", ms=4,
                               label="Det. rate")],
               fontsize=9, loc="lower right")
    fig.tight_layout()
    tag = f"{regime}_a{_ALPHA_STR[alpha].replace('.', '')}"
    save(fig, fig_path("detection", f"accept_acc_vs_threshold_{tag}.pdf", arch=ARCH))

## §2.6  Gibbs purification against itself

Gibbs purification is **attack-radius agnostic**: its `step_radius` (0.1 here, i.e. absolute 0.2) is a *per-sweep*
$L_\infty$ move that re-centres each sweep, so strength is controlled by the sweep count $k$
alone and results are keyed by $k$, never by a radius.

All three figures share the same $k=0$ convention: the anchor is the **undefended** accuracy
measured on the *same* 250-sample subsample (`gibbs_adv_acc/{eps}` for adversarial input,
`gibbs_clean_acc` for clean), so the $k{=}0 \to k{>}0$ lift is paired and the sampling noise is
common-mode.

> **On the headline $k$.** The available sweep counts are $k \in \{1, 3, 6\}$ — `analysis/gibbs.py`
> was run with `--sweeps 1,3,6`, so **there is no $k=5$ in this data**. `GIBBS_K` is set to **6**,
> the nearest available. To get a real $k=5$, re-run
> `python analysis/gibbs.py <sweep_dir> --sweeps 1,3,5 --n-eval 250` for every run dir and then
> set `GIBBS_K = 5` in §0.2 — the guard printed there checks the constant against the CSVs.

- **§2.6a** — one line per model at $\varepsilon = $ `EPS_MAIN`.
- **§2.6b** — the same, one panel per $\varepsilon$.
- **§2.6c** — the clean-input counterpart: what purification *costs* when there is no attack.
  A defense that recovers adversarial accuracy but destroys clean accuracy is not a defense.

Both AT variants appear here: `at/…/a001` has a `gibbs_data.csv` but **no**
`evaluation_data.csv`, so it is present in these three figures and absent from every other
figure and table in Part 2.

In [ ]:
gmodels = [(r, a) for r, a in MODEL_ORDER if not sel(GIBBS, regime=r, arch=ARCH, alpha=a).empty]

fig, ax = plt.subplots(figsize=(7, 4.6))
for regime, alpha in gmodels:
    k, m, s = gibbs_curve(sel(GIBBS, regime=regime, arch=ARCH, alpha=alpha), EPS_MAIN)
    errline(ax, k, m, s, MODEL_COLOR[(regime, alpha)], model_label(regime, alpha))
ax.set_xticks([0] + GIBBS_KS)
ax.set_xlabel("Gibbs sweeps $k$   ($k=0$: undefended, same subsample)")
ax.set_ylabel("Accuracy under attack")
ax.set_ylim(0, 1.02); ax.grid(True, alpha=0.3)
ax.set_title(rf"{ARCH} — Gibbs purification ($\varepsilon={EPS_MAIN}$, "
             rf"$n_{{\mathrm{{eval}}}}={GIBBS_N_EVAL}$)")
ax.legend(fontsize=9, ncol=2, loc="best")
fig.tight_layout()
save(fig, fig_path("gibbs", "gibbs_alpha_vs_k.pdf", arch=ARCH))

In [ ]:
fig, axes = plt.subplots(1, len(EPS_LIST), figsize=(4.7 * len(EPS_LIST), 4.3),
                         sharey=True, squeeze=False)
for ax, eps in zip(axes[0], EPS_LIST):
    for regime, alpha in gmodels:
        k, m, s = gibbs_curve(sel(GIBBS, regime=regime, arch=ARCH, alpha=alpha), eps)
        errline(ax, k, m, s, MODEL_COLOR[(regime, alpha)], model_label(regime, alpha))
    ax.set_title(rf"$\varepsilon={eps}$")
    ax.set_xticks([0] + GIBBS_KS); ax.set_xlabel("Gibbs sweeps $k$")
    ax.set_ylim(0, 1.02); ax.grid(True, alpha=0.3)
axes[0][0].set_ylabel("Accuracy under attack")
axes[0][0].legend(fontsize=8, ncol=2, loc="best")
fig.suptitle(rf"{ARCH} — Gibbs purification vs $k$ "
             rf"($n_{{\mathrm{{eval}}}}={GIBBS_N_EVAL}$)", fontsize=15)
fig.tight_layout()
save(fig, fig_path("gibbs", "gibbs_alpha_vs_k_by_eps.pdf", arch=ARCH))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.4))
for regime, alpha in gmodels:
    k, m, s = gibbs_curve(sel(GIBBS, regime=regime, arch=ARCH, alpha=alpha), eps=None)
    errline(ax, k, m, s, MODEL_COLOR[(regime, alpha)], model_label(regime, alpha))
ax.set_xticks([0] + GIBBS_KS)
ax.set_xlabel("Gibbs sweeps $k$   ($k=0$: unpurified clean input)")
ax.set_ylabel("Clean accuracy")
ax.grid(True, alpha=0.3)
ax.set_title(rf"{ARCH} — cost of Gibbs purification on clean input "
             rf"($n_{{\mathrm{{eval}}}}={GIBBS_N_EVAL}$)")
ax.legend(fontsize=9, ncol=2, loc="best")
fig.tight_layout()
save(fig, fig_path("gibbs", "gibbs_clean_cost.pdf", arch=ARCH))

## §2.7  Likelihood purification — the two radii

Purification accuracy against attack budget, one line per purification radius
($r = 0.2$ and $r = 0.3$, absolute values in the embedding domain), faceted over every model
with an `evaluation_data.csv`. The dashed grey reference is undefended `rob/{eps}`, so the
vertical gap is the purification lift.

The larger radius searches a bigger ball for a higher-likelihood point: it should recover more
under strong attack, but risks over-moving the input under weak attack. The `PURIFY_RADIUS`
used by §2.2/§2.4/§2.8 is set in §0.2.

In [ ]:
models = [(r, a) for r, a in MODEL_ORDER if not sel(EVAL, regime=r, arch=ARCH, alpha=a).empty]
RADIUS_COLOR = {"0.2": "#4CAF50", "0.3": "#1B5E20"}

ncol = min(len(models), 4)
nrow = int(np.ceil(len(models) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.0 * ncol, 3.7 * nrow),
                         sharey=True, sharex=True, squeeze=False)
for ax, (regime, alpha) in zip(axes.ravel(), models):
    df = sel(EVAL, regime=regime, arch=ARCH, alpha=alpha)
    rob = [ms(df, f"rob/{e}")[0] for e in EPS_LIST]
    ax.plot(EPS_LIST, rob, color="grey", ls="--", lw=1.4, marker="x", ms=5,
            label="Rob. (no def.)")
    for r in PURIFY_RADII:
        stats = [ms(df, f"uq_purify_acc/{e}/{r}") for e in EPS_LIST]
        m, s = map(np.array, zip(*stats))
        if np.all(np.isnan(m)):
            continue
        errline(ax, np.array(EPS_LIST), m, np.nan_to_num(s), RADIUS_COLOR[r], rf"$r{{=}}{r}$")
    ax.set_title(model_label(regime, alpha))
    ax.set_ylim(0, 1.02); ax.grid(True, alpha=0.3)
for ax in axes.ravel()[len(models):]:
    ax.axis("off")
for ax in axes[-1, :]:
    ax.set_xlabel(r"attack $\varepsilon$")
for ax in axes[:, 0]:
    ax.set_ylabel("Purified accuracy")
axes[0, 0].legend(fontsize=9, loc="upper right")
fig.suptitle(f"{ARCH} — likelihood purification radius", fontsize=15)
fig.tight_layout()
save(fig, fig_path("purify_radius", "purify_acc_vs_radius.pdf", arch=ARCH))

## §2.8  Model-comparison tables

Two tables for the committed bond dimension, each as booktabs `.tex`
(needs `\usepackage{booktabs,multirow}`) and an aligned `.txt` preview. Best per column in
**bold** (`*` in the `.txt`); all seven metrics are higher-is-better.

- **`model_comparison_eps{EPS_MAIN}`** — the compact headline table: one row per model, one
  column per metric, at a single $\varepsilon$.
- **`model_comparison`** — the full table: metric rows nested under each model, one column per
  $\varepsilon$.

Caveat carried in both captions: the Gibbs column is a 250-sample estimate, every other column
uses the full test split.

In [ ]:
# (tex header, txt header, source, column template, flip)
TABLE_METRICS = [
    ("Clean",                                    "Clean",        EVAL,  "acc",                                       False),
    (r"Rob.",                                    "Rob.",         EVAL,  "rob/{eps}",                                 False),
    (r"Purif.\ (lk., $r{=}0.2$)",                "Purif lk r.2", EVAL,  "uq_purify_acc/{eps}/0.2",                   False),
    (rf"Purif.\ (Gibbs, $k{{=}}{GIBBS_K}$)",     f"Purif Gk{GIBBS_K}",
                                                                 GIBBS, f"gibbs_purify_acc/{{eps}}/{GIBBS_K}",       False),
    (r"Det.\ rate ($q{=}10\%$)",                 "Det rate",     EVAL,  f"uq_detection/{DET_PCT}/{{eps}}",           False),
    (r"Acc.\ passed ($q{=}10\%$)",               "Acc passed",   EVAL,  f"uq_det_err_passed/{DET_PCT}/{{eps}}",      True),
]

TABLE_MODELS = [(r, a) for r, a in MODEL_ORDER
                if not sel(EVAL, regime=r, arch=ARCH, alpha=a).empty]
CAPTION_NOTE = (rf"MNIST $12\times12$, Legendre $d{{=}}3$, $r{{=}}{BOND_DIM}$, 5 seeds "
                rf"(mean $\pm$ 1 std). Gibbs uses an "
                rf"$n_{{\mathrm{{eval}}}}={GIBBS_N_EVAL}$ subsample; all other columns use the "
                rf"full test split.")


def cellstat(regime, alpha, src, tmpl, flip, eps):
    return ms(sel(src, regime=regime, arch=ARCH, alpha=alpha), tmpl.format(eps=eps), flip)


# ---- compact table: models x metrics, single epsilon ----------------------
grid = [[cellstat(r, a, src, tmpl, flip, EPS_MAIN)
         for _, _, src, tmpl, flip in TABLE_METRICS] for r, a in TABLE_MODELS]
best = [best_index([grid[i][j][0] for i in range(len(TABLE_MODELS))])
        for j in range(len(TABLE_METRICS))]

L = [r"\begin{table}[htbp]", r"  \centering", r"  \small",
     r"  \begin{tabular}{@{}l" + "r" * len(TABLE_METRICS) + r"@{}}", r"    \toprule",
     "    Model & " + " & ".join(h for h, *_ in TABLE_METRICS) + r" \\", r"    \midrule"]
for i, (r_, a) in enumerate(TABLE_MODELS):
    L.append(f"    {model_label(r_, a)} & " +
             " & ".join(tex_cell(m, s, bold=(best[j] == i))
                        for j, (m, s) in enumerate(grid[i])) + r" \\")
L += [r"    \bottomrule", r"  \end{tabular}",
      rf"  \caption{{Defenses at $\varepsilon={EPS_MAIN}$, $q{{=}}10\%$. {CAPTION_NOTE}}}",
      rf"  \label{{tab:mnist_r12_models_r{BOND_DIM}_eps{EPS_MAIN}}}", r"\end{table}"]
fig_path("tables", f"model_comparison_eps{EPS_MAIN}.tex", arch=ARCH).write_text("\n".join(L) + "\n")

W = 17
hdr = f"{'Model':<14}" + "".join(f"{h2:<{W}}" for _, h2, *_ in TABLE_METRICS)
P = [f"{ARCH} — defenses at eps={EPS_MAIN}, q=10% (mean±std over seeds); * = best per column",
     f"Gibbs column: n_eval={GIBBS_N_EVAL} subsample; all others: full test split",
     hdr, "-" * len(hdr)]
for i, (r_, a) in enumerate(TABLE_MODELS):
    P.append(f"{model_label(r_, a, tex=False):<14}" +
             "".join(f"{txt_cell(m, s, best=(best[j] == i)):<{W}}"
                     for j, (m, s) in enumerate(grid[i])))
fig_path("tables", f"model_comparison_eps{EPS_MAIN}.txt", arch=ARCH).write_text("\n".join(P) + "\n")
print("\n".join(P))
print()

In [ ]:
# ---- full table: (model x metric) rows, epsilon columns -------------------
data = [[[cellstat(r_, a, src, tmpl, flip, e) for e in EPS_LIST]
         for _, _, src, tmpl, flip in TABLE_METRICS] for r_, a in TABLE_MODELS]
best = [[best_index([data[i][j][e][0] for i in range(len(TABLE_MODELS))])
         for e in range(len(EPS_LIST))] for j in range(len(TABLE_METRICS))]

n_m = len(TABLE_METRICS)
L = [r"\begin{table}[htbp]", r"  \centering", r"  \small",
     r"  \begin{tabular}{@{}ll" + "r" * len(EPS_LIST) + r"@{}}", r"    \toprule",
     "    Model & Metric & " + " & ".join(rf"$\varepsilon={e}$" for e in EPS_LIST) + r" \\",
     r"    \midrule"]
for i, (r_, a) in enumerate(TABLE_MODELS):
    for j, (h, *_rest) in enumerate(TABLE_METRICS):
        first = f"\\multirow{{{n_m}}}{{*}}{{{model_label(r_, a)}}}" if j == 0 else ""
        cells = [tex_cell(*data[i][j][e], bold=(best[j][e] == i)) for e in range(len(EPS_LIST))]
        L.append(f"    {first} & {h} & " + " & ".join(cells) + r" \\")
    if i < len(TABLE_MODELS) - 1:
        L.append(r"    \cmidrule(lr){1-%d}" % (2 + len(EPS_LIST)))
L += [r"    \bottomrule", r"  \end{tabular}",
      rf"  \caption{{Defenses across attack budgets, $q{{=}}10\%$, likelihood-purification "
      rf"radii as labelled. {CAPTION_NOTE}}}",
      rf"  \label{{tab:mnist_r12_models_r{BOND_DIM}}}", r"\end{table}"]
fig_path("tables", "model_comparison.tex", arch=ARCH).write_text("\n".join(L) + "\n")

W = 17
hdr = f"{'Model':<14}{'Metric':<16}" + "".join(f"{'eps=' + str(e):<{W}}" for e in EPS_LIST)
P = [f"{ARCH} — defenses vs attack budget, q=10% (mean±std over seeds); * = best per column",
     f"Gibbs rows: n_eval={GIBBS_N_EVAL} subsample; all others: full test split",
     hdr, "-" * len(hdr)]
for i, (r_, a) in enumerate(TABLE_MODELS):
    for j, (_, h2, *_rest) in enumerate(TABLE_METRICS):
        lab = model_label(r_, a, tex=False) if j == 0 else ""
        P.append(f"{lab:<14}{h2:<16}" +
                 "".join(f"{txt_cell(*data[i][j][e], best=(best[j][e] == i)):<{W}}"
                         for e in range(len(EPS_LIST))))
    P.append("-" * len(hdr))
fig_path("tables", "model_comparison.txt", arch=ARCH).write_text("\n".join(P) + "\n")
print("\n".join(P))

## §2.9  Adaptive (joint) attack — difference table

$\Delta = $ joint $-$ standard, one table per $\varepsilon$. The joint attack is
detector-aware: it maximises classification loss *while* keeping $\log p(x)$ high, so a
negative $\Delta$ means the adaptive attacker is stronger (lower accuracy, or lower detection
rate). This is the honesty check on §2.5 — a detector that only works against a
detector-unaware attacker is not a defense.

`Det.` uses the detection **rate** rather than error-among-detected, which goes `NaN` once the
joint attack fully evades the detector.

In [ ]:
# (joint template, standard template, tex label, txt label)
JN_METRICS = [
    ("uq_joint_adv_acc/{eps}", "uq_adv_acc/{eps}", r"Rob.", "Rob."),
    (f"uq_joint_purify_acc/{{eps}}/{PURIFY_RADIUS}", f"uq_purify_acc/{{eps}}/{PURIFY_RADIUS}",
     r"Purif.\ (lk.)", "Purif lk"),
    (f"uq_joint_detection/{DET_PCT}/{{eps}}", f"uq_detection/{DET_PCT}/{{eps}}",
     r"Det.\ rate ($q{=}10\%$)", "Det rate"),
    (f"uq_joint_det_err_passed/{DET_PCT}/{{eps}}", f"uq_det_err_passed/{DET_PCT}/{{eps}}",
     r"Accept.\ err ($q{=}10\%$)", "Accept err"),
]


def delta(df, jcol, ncol):
    """Paired per-seed difference (joint - standard); (nan, nan) if either column is absent."""
    if df.empty or jcol not in df.columns or ncol not in df.columns:
        return float("nan"), float("nan")
    d = (pd.to_numeric(df[jcol], errors="coerce").to_numpy()
         - pd.to_numeric(df[ncol], errors="coerce").to_numpy())
    d = d[np.isfinite(d)]
    if len(d) == 0:
        return float("nan"), float("nan")
    return float(d.mean()), float(d.std() if len(d) > 1 else 0.0)


dfs = [(model_label(r, a), model_label(r, a, tex=False),
        sel(EVAL, regime=r, arch=ARCH, alpha=a)) for r, a in TABLE_MODELS]
plain_all = []
for eps in EPS_LIST:
    L = [r"\begin{table}[htbp]", r"  \centering",
         r"  \begin{tabular}{@{}l" + "c" * len(dfs) + r"@{}}", r"    \toprule",
         "    Metric & " + " & ".join(t for t, _, _ in dfs) + r" \\", r"    \midrule"]
    plain = [f"=== eps={eps} ===",
             f"{'Metric':<14}" + "".join(f"{p:<18}" for _, p, _ in dfs)]
    for jt, nt, tlab, plab in JN_METRICS:
        cells = [delta(df, jt.format(eps=eps), nt.format(eps=eps)) for _, _, df in dfs]
        L.append(f"    {tlab} & " +
                 " & ".join(tex_cell(m, s, signed=True) for m, s in cells) + r" \\")
        plain.append(f"{plab:<14}" +
                     "".join(f"{txt_cell(m, s, signed=True):<18}" for m, s in cells))
    L += [r"    \bottomrule", r"  \end{tabular}",
          rf"  \caption{{Joint (detector-aware) minus standard attack at $\varepsilon={eps}$, "
          rf"$q{{=}}10\%$. Negative $\Rightarrow$ the joint attack is stronger. {CAPTION_NOTE}}}",
          rf"  \label{{tab:mnist_r12_joint_r{BOND_DIM}_eps{eps}}}", r"\end{table}"]
    fig_path("tables", f"joint_vs_normal_eps{eps}.tex", arch=ARCH).write_text("\n".join(L) + "\n")
    plain_all += plain + [""]

fig_path("tables", "joint_vs_normal.txt", arch=ARCH).write_text("\n".join(plain_all) + "\n")
print(f"Saved joint_vs_normal_eps{{{','.join(str(e) for e in EPS_LIST)}}}.tex "
      f"and joint_vs_normal.txt\n")
print("\n".join(plain_all))

---
## Where everything landed

```
figures/mnist_r12/
  compare/
    bonddim/  metric_vs_bonddim.png · alpha_curve_by_bonddim.png
              accept_curve_by_bonddim.png · nll_vs_bonddim.png
    tables/   bonddim_summary.{tex,txt}
  {ARCH}/
    sampling/       mnist_samples.png
    alpha/          alpha_curve_accuracy.png · alpha_curve_nll.png
    robustness/     defense_comparison_eps{0.1,0.2,0.3}.png
    detection/      accept_acc_vs_passed_ratio.png
                    accept_acc_vs_threshold_{nat_a0,nat_a001,nat_a1,at_a0}.png
    gibbs/          gibbs_alpha_vs_k.png · gibbs_alpha_vs_k_by_eps.png
                    gibbs_clean_cost.png
    purify_radius/  purify_acc_vs_radius.png
    tables/         model_comparison.{tex,txt}
                    model_comparison_eps{EPS_MAIN}.{tex,txt}
                    joint_vs_normal_eps{0.1,0.2,0.3}.tex · joint_vs_normal.txt
```

To regenerate Part 2 at another bond dimension, set `BOND_DIM = 10` (or `40`) in §0.2 and
re-run from there.